# Part 1: Customer 360 Feature Engineering - Data Preparation

## Objective
The objective of this step is to prepare all required datasets needed to create a unified customer-level intelligence dataset.

Customer 360 aims to combine customer information, order history, payment behavior, review experience, and product purchase patterns into a single customer profile.

## Data Sources Used

The following Olist e-commerce datasets are used:

- **Customers Data**
  - Customer identifier
  - Customer location details

- **Orders Data**
  - Order history
  - Order status
  - Purchase dates
  - Delivery information

- **Payments Data**
  - Customer spending behavior
  - Payment methods
  - Installment details

- **Reviews Data**
  - Customer satisfaction and review scores

- **Order Products Data**
  - Product purchase behavior
  - Items and product diversity

## Process

1. Load all required processed datasets.
2. Create safe copies of original datasets to avoid data loss.
3. Validate dataset size and structure.
4. Prepare datasets for customer-level feature aggregation.

## Outcome

All required data sources are prepared for building the Customer 360 dataset containing complete customer behavior information.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [2]:
customers = pd.read_csv("../1 data/02_processed data/customers_clean.csv")

orders = pd.read_csv(
    "../1 data/02_processed data/orders_clean.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

order_items = pd.read_csv(
    "../1 data/02_processed data/order_items_clean.csv",
    parse_dates=["shipping_limit_date"]
)

payments = pd.read_csv("../1 data/02_processed data/payments_clean.csv")

reviews = pd.read_csv(
    "../1 data/02_processed data/reviews_clean.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

products = pd.read_csv("../1 data/02_processed data/products_clean.csv")

In [3]:
customers_df = customers.copy()

orders_df = orders.copy()

order_items_df = order_items.copy()

payments_df = payments.copy()

reviews_df = reviews.copy()

products_df = products.copy()

In [4]:
datasets = {
    "Customers": customers_df,
    "Orders": orders_df,
    "Order Items": order_items_df,
    "Payments": payments_df,
    "Reviews": reviews_df,
    "Products": products_df
}

for name, df in datasets.items():
    print(f"{name:<15} : {df.shape}")

Customers       : (99441, 5)
Orders          : (99441, 8)
Order Items     : (112650, 7)
Payments        : (103886, 5)
Reviews         : (99224, 7)
Products        : (32951, 9)


In [5]:
order_items_df.info()
payments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 16.3 MB
<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  in

In [6]:
reviews_df.info()
products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  str           
 4   review_comment_message   40977 non-null  str           
 5   review_creation_date     99224 non-null  datetime64[us]
 6   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 14.2 MB
<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_categor

In [7]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    display(df.head())


Customers


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



Orders


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26



Order Items


...

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45



Reviews


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53



Products


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [8]:
# select required columns
customer_profile = customers_df[
    [
        "customer_unique_id",
        "customer_city",
        "customer_state"
    ]
].copy()

In [9]:
#check dataset shape
customer_profile.shape

(99441, 3)

In [10]:
#Remove Duplicate Customers
customer_profile = customer_profile.drop_duplicates(
    subset="customer_unique_id"
).reset_index(drop=True)

In [11]:
#Validate Dataset
print("Shape :", customer_profile.shape)
print("Unique Customers :", customer_profile["customer_unique_id"].nunique())
print("Duplicate Customers :", customer_profile.duplicated().sum())

Shape : (96096, 3)
Unique Customers : 96096
Duplicate Customers : 0


In [12]:
customer_profile.head()

,customer_unique_id,customer_city,customer_state
0,861eff4711a542e4b93843c6dd7febb0,franca,SP
1,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP
2,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP
3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP
4,345ecd01c38d18a9036ed96c73b8d066,campinas,SP


In [13]:
#  merge orders with customers
orders_customer = orders_df.merge(
    customers_df[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)
orders_customer.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6


In [14]:
# create purchase behaviour feature
purchase_behavior = (
    orders_customer
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "count"),
        delivered_orders=("order_status", lambda x: (x == "delivered").sum()),
        cancelled_orders=("order_status", lambda x: (x == "canceled").sum()),
        first_purchase_date=("order_purchase_timestamp", "min"),
        last_purchase_date=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

In [15]:
#Validate Dataset
print("Shape :", purchase_behavior.shape)

print("Unique Customers :", purchase_behavior["customer_unique_id"].nunique())

print("Duplicate Customers :", purchase_behavior.duplicated().sum())

Shape : (96096, 6)
Unique Customers : 96096
Duplicate Customers : 0


In [16]:
purchase_behavior.head()

,customer_unique_id,total_orders,delivered_orders,cancelled_orders,first_purchase_date,last_purchase_date
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,0,2018-05-10 10:56:27,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,0,2018-05-07 11:11:27,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,1,1,0,2017-03-10 21:05:03,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,0,2017-10-12 20:29:41,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,1,1,0,2017-11-14 19:45:42,2017-11-14 19:45:42


In [17]:
#  Payment Behaviour Features
# Merge Payments with Customer Information
payments_customer = (
    payments_df
    .merge(
        orders_df[["order_id", "customer_id"]],
        on="order_id",
        how="left"
    )
    .merge(
        customers_df[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left"
    )
)

In [18]:

#Step 2: Create Payment Behaviour Features
payment_behavior = (
    payments_customer
    .groupby("customer_unique_id")
    .agg(
        total_spent=("payment_value", "sum"),
        average_order_value=("payment_value", "mean"),
        maximum_order_value=("payment_value", "max"),
        minimum_order_value=("payment_value", "min"),
        average_payment_installments=("payment_installments", "mean")
    )
    .reset_index()
)


In [19]:
#step 3 preferred payment mwthod
preferred_payment = (
    payments_customer
    .groupby("customer_unique_id")["payment_type"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    .reset_index(name="preferred_payment_type")
)
preferred_payment.head()

,customer_unique_id,preferred_payment_type
0,0000366f3b9a7992bf8c76cfdf3221e2,credit_card
1,0000b849f77a49e4a4ce2b2a4ca5be3f,credit_card
2,0000f46a3911fa3c0805444483337064,credit_card
3,0000f6ccb0745a6a4b88665a16c9f078,credit_card
4,0004aac84e0df4da2b147fca70cf8255,credit_card


In [20]:
#step 4 merge prefered payment method

payment_behavior = payment_behavior.merge(
    preferred_payment,
    on="customer_unique_id",
    how="left"
)

In [21]:
#validate dataset
print("Shape :", payment_behavior.shape)

print("Unique Customers :", payment_behavior["customer_unique_id"].nunique())

print("Duplicate Customers :", payment_behavior.duplicated().sum())

Shape : (96095, 7)
Unique Customers : 96095
Duplicate Customers : 0


In [22]:
#
# preview
payment_behavior.head()

,customer_unique_id,total_spent,average_order_value,maximum_order_value,minimum_order_value,average_payment_installments,preferred_payment_type
0,0000366f3b9a7992bf8c76cfdf3221e2,141.90,141.90,141.90,141.90,8.0,credit_card
1,0000b849f77a49e4a4ce2b2a4ca5be3f,27.19,27.19,27.19,27.19,1.0,credit_card
2,0000f46a3911fa3c0805444483337064,86.22,86.22,86.22,86.22,8.0,credit_card
3,0000f6ccb0745a6a4b88665a16c9f078,43.62,43.62,43.62,43.62,4.0,credit_card
4,0004aac84e0df4da2b147fca70cf8255,196.89,196.89,196.89,196.89,6.0,credit_card


In [23]:
#investigation why unique customer value is 96095
print(payments_customer.shape)
print(payments_customer.columns)

print(payments_customer["customer_unique_id"].nunique())

(103886, 7)
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'customer_id',
       'customer_unique_id'],
      dtype='str')
96095


In [24]:
payments_customer["customer_unique_id"].isnull().sum()

np.int64(0)

In [25]:
print(customers_df["customer_unique_id"].nunique())

96096


In [26]:
missing_customers = set(customers_df["customer_unique_id"]) - set(payments_customer["customer_unique_id"])

print(len(missing_customers))
print(missing_customers)

1
{'830d5b7aaa3b6f1e9ad63703bec97d23'}


# Note:
 One customer is missing because no payment record exists in the Payments dataset.
 This is an original dataset characteristic, not a merge error.

In [27]:
#Part 5 – Product Behaviour Features
#Step 1: Merge Required Datasets
product_customer = (
    order_items_df
    .merge(
        orders_df[["order_id", "customer_id"]],
        on="order_id",
        how="left"
    )
    .merge(
        customers_df[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left"
    )
    .merge(
        products_df[["product_id", "product_category_name"]],
        on="product_id",
        how="left"
    )
)

In [28]:
#Step 2: Create Product Behaviour Features
product_behavior = (
    product_customer
    .groupby("customer_unique_id")
    .agg(
        total_items_purchased=("order_item_id", "count"),
        unique_products=("product_id", "nunique"),
        unique_categories=("product_category_name", "nunique")
    )
    .reset_index()
)
print(product_behavior.head())

                 customer_unique_id  total_items_purchased  unique_products  \
0  0000366f3b9a7992bf8c76cfdf3221e2                      1                1   
1  0000b849f77a49e4a4ce2b2a4ca5be3f                      1                1   
2  0000f46a3911fa3c0805444483337064                      1                1   
3  0000f6ccb0745a6a4b88665a16c9f078                      1                1   
4  0004aac84e0df4da2b147fca70cf8255                      1                1   

   unique_categories  
0                  1  
1                  1  
2                  1  
3                  1  
4                  1  


In [29]:
#Step 3: Favourite Product Category
favorite_category = (
    product_customer
    .groupby("customer_unique_id")["product_category_name"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    .reset_index(name="favorite_category")
)
print(favorite_category.head())

                 customer_unique_id favorite_category
0  0000366f3b9a7992bf8c76cfdf3221e2   cama_mesa_banho
1  0000b849f77a49e4a4ce2b2a4ca5be3f      beleza_saude
2  0000f46a3911fa3c0805444483337064         papelaria
3  0000f6ccb0745a6a4b88665a16c9f078         telefonia
4  0004aac84e0df4da2b147fca70cf8255         telefonia


In [30]:
#Step 4: Average Items Per Order
avg_items = (
    product_customer
    .groupby(["customer_unique_id", "order_id"])
    .size()
    .reset_index(name="items_per_order")
)

avg_items = (
    avg_items
    .groupby("customer_unique_id")["items_per_order"]
    .mean()
    .reset_index(name="average_items_per_order")
)
print(avg_items.head())

                 customer_unique_id  average_items_per_order
0  0000366f3b9a7992bf8c76cfdf3221e2                      1.0
1  0000b849f77a49e4a4ce2b2a4ca5be3f                      1.0
2  0000f46a3911fa3c0805444483337064                      1.0
3  0000f6ccb0745a6a4b88665a16c9f078                      1.0
4  0004aac84e0df4da2b147fca70cf8255                      1.0


In [41]:
purchase_behavior.head()

,customer_unique_id,total_orders,delivered_orders,cancelled_orders,first_purchase_date,last_purchase_date
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,0,2018-05-10 10:56:27,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,0,2018-05-07 11:11:27,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,1,1,0,2017-03-10 21:05:03,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,0,2017-10-12 20:29:41,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,1,1,0,2017-11-14 19:45:42,2017-11-14 19:45:42


In [31]:
#Step 5: Merge All Product Features
product_behavior = (
    product_behavior
    .merge(
        favorite_category,
        on="customer_unique_id",
        how="left"
    )
    .merge(
        avg_items,
        on="customer_unique_id",
        how="left"
    )
)

In [32]:
#Step 6: Validate Dataset
print("Shape :", product_behavior.shape)

print("Unique Customers :", product_behavior["customer_unique_id"].nunique())

print("Duplicate Customers :", product_behavior.duplicated().sum())

Shape : (95420, 6)
Unique Customers : 95420
Duplicate Customers : 0


In [33]:
# investigation
missing_customers = (
    set(customers_df["customer_unique_id"])
    - set(product_behavior["customer_unique_id"])
)

print(len(missing_customers))

676


# Note:
 676 customers do not have product-level records in the Order Items dataset.
 They will be retained in the final Customer 360 dataset using a left join.

In [34]:
product_behavior.head()

,customer_unique_id,total_items_purchased,unique_products,unique_categories,favorite_category,average_items_per_order
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,1,cama_mesa_banho,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,1,beleza_saude,1.0
2,0000f46a3911fa3c0805444483337064,1,1,1,papelaria,1.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,1,telefonia,1.0
4,0004aac84e0df4da2b147fca70cf8255,1,1,1,telefonia,1.0


In [35]:
#Part 6 – Customer Experience Features
#Step 1: Merge Required Datasets
experience_customer = (
    orders_df
    .merge(
        customers_df[["customer_id", "customer_unique_id"]],
        on="customer_id",
        how="left"
    )
    .merge(
        reviews_df[["order_id", "review_score"]],
        on="order_id",
        how="left"
    )
)

In [36]:
#Step 2: Create Delivery Features
experience_customer["delivery_days"] = (
    experience_customer["order_delivered_customer_date"]
    - experience_customer["order_purchase_timestamp"]
).dt.days

experience_customer["delivery_delay"] = (
    experience_customer["order_delivered_customer_date"]
    - experience_customer["order_estimated_delivery_date"]
).dt.days

In [37]:
#Step 3: Create On-Time Delivery Flag
experience_customer["on_time_delivery"] = (
    experience_customer["delivery_delay"] <= 0
).astype(int)

In [38]:
#Step 4: Create Customer Experience Features
experience_behavior = (
    experience_customer
    .groupby("customer_unique_id")
    .agg(
        average_review_score=("review_score", "mean"),
        average_delivery_days=("delivery_days", "mean"),
        average_delivery_delay=("delivery_delay", "mean"),
        on_time_delivery_rate=("on_time_delivery", "mean")
    )
    .reset_index()
)

In [39]:
#Step 5: Delayed Delivery Rate
experience_behavior["delayed_delivery_rate"] = (
    1 - experience_behavior["on_time_delivery_rate"]
)

In [40]:
#Step 6: Validate Dataset
print("Shape :", experience_behavior.shape)

print("Unique Customers :", experience_behavior["customer_unique_id"].nunique())

print("Duplicate Customers :", experience_behavior.duplicated().sum())

Shape : (96096, 6)
Unique Customers : 96096
Duplicate Customers : 0


In [42]:
experience_behavior.head()

,customer_unique_id,average_review_score,average_delivery_days,average_delivery_delay,on_time_delivery_rate,delayed_delivery_rate
0,0000366f3b9a7992bf8c76cfdf3221e2,5.0,6.0,-5.0,1.0,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,4.0,3.0,-5.0,1.0,0.0
2,0000f46a3911fa3c0805444483337064,3.0,25.0,-2.0,1.0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,4.0,20.0,-12.0,1.0,0.0
4,0004aac84e0df4da2b147fca70cf8255,5.0,13.0,-8.0,1.0,0.0


In [43]:
#Part 7 – Advanced Behaviour Features
#Step 1: Merge Required Feature Tables
advanced_behavior = purchase_behavior.merge(
    payment_behavior,
    on="customer_unique_id",
    how="left"
)

In [44]:
#Step 2: Create Snapshot Date
snapshot_date = (
    advanced_behavior["last_purchase_date"].max()
    + pd.Timedelta(days=1)
)

In [45]:
#Step 3: Recency
advanced_behavior["recency_days"] = (
    snapshot_date
    - advanced_behavior["last_purchase_date"]
).dt.days

In [46]:
#Step 4: Customer Tenure
advanced_behavior["customer_tenure_days"] = (
    advanced_behavior["last_purchase_date"]
    - advanced_behavior["first_purchase_date"]
).dt.days

In [47]:
#Step 5: Purchase Frequency
advanced_behavior["purchase_frequency"] = (
    advanced_behavior["total_orders"]
    / (advanced_behavior["customer_tenure_days"] + 1)
).round(2)

In [48]:
#Step 6: Average Purchase Gap
advanced_behavior["average_purchase_gap"] = (
    advanced_behavior["customer_tenure_days"]
    / advanced_behavior["total_orders"]
).round(2)

In [49]:
#Step 7: Spending Intensity
advanced_behavior["spending_intensity"] = (
    advanced_behavior["total_spent"]
    / advanced_behavior["total_orders"]
).round(2)

In [50]:
#Step 8: Repeat Purchase Flag
advanced_behavior["repeat_customer"] = (
    advanced_behavior["total_orders"] > 1
).astype(int)

In [51]:
#Step 9: Cancel Rate
advanced_behavior["cancel_rate"] = (
    advanced_behavior["cancelled_orders"]
    / advanced_behavior["total_orders"]
).round(2)

In [53]:
#Step 10: Validate Dataset
print("Shape :", advanced_behavior.shape)

print("Unique Customers :", advanced_behavior["customer_unique_id"].nunique())

print("Duplicate Customers :", advanced_behavior.duplicated().sum())
print(advanced_behavior.head())

Shape : (96096, 19)
Unique Customers : 96096
Duplicate Customers : 0
                 customer_unique_id  total_orders  delivered_orders  \
0  0000366f3b9a7992bf8c76cfdf3221e2             1                 1   
1  0000b849f77a49e4a4ce2b2a4ca5be3f             1                 1   
2  0000f46a3911fa3c0805444483337064             1                 1   
3  0000f6ccb0745a6a4b88665a16c9f078             1                 1   
4  0004aac84e0df4da2b147fca70cf8255             1                 1   

   cancelled_orders first_purchase_date  last_purchase_date  total_spent  \
0                 0 2018-05-10 10:56:27 2018-05-10 10:56:27       141.90   
1                 0 2018-05-07 11:11:27 2018-05-07 11:11:27        27.19   
2                 0 2017-03-10 21:05:03 2017-03-10 21:05:03        86.22   
3                 0 2017-10-12 20:29:41 2017-10-12 20:29:41        43.62   
4                 0 2017-11-14 19:45:42 2017-11-14 19:45:42       196.89   

   average_order_value  maximum_order_value  mi

In [54]:
#Part 8 – Business Intelligence Features
#Step 1: Create Business Intelligence Table
business_intelligence = advanced_behavior.copy()


In [55]:
#Step 2: One-Time Buyer Flag
business_intelligence["one_time_buyer"] = (
    business_intelligence["total_orders"] == 1
).astype(int)

In [56]:
#Step 3: Repeat Customer Flag
business_intelligence["repeat_customer"] = (
    business_intelligence["total_orders"] > 1
).astype(int)

In [57]:
#Step 4: High Spender Flag


median_spending = business_intelligence["total_spent"].median()

business_intelligence["high_spender"] = (
    business_intelligence["total_spent"] > median_spending
).astype(int)

In [58]:
#Step 5: Frequent Buyer Flag
business_intelligence["vip_customer"] = (
    (business_intelligence["high_spender"] == 1)
    &
    (business_intelligence["repeat_customer"] == 1)
).astype(int)

In [59]:
#Step 6: VIP Customer Flag
business_intelligence["vip_customer"] = (
    (business_intelligence["high_spender"] == 1)
    &
    (business_intelligence["repeat_customer"] == 1)
).astype(int)

In [60]:
#Step 7: Loyal Customer Flag
business_intelligence["loyal_customer"] = (
    (business_intelligence["repeat_customer"] == 1)
    &
    (business_intelligence["recency_days"] <= 90)
).astype(int)

In [61]:
#Step 8: At-Risk Customer Flag
business_intelligence["at_risk_customer"] = (
    business_intelligence["recency_days"] > 180
).astype(int)

In [62]:
#Step 9: Customer Value Tier
business_intelligence["customer_value_tier"] = pd.qcut(
    business_intelligence["total_spent"],
    q=3,
    labels=["Low", "Medium", "High"]
)

In [63]:
#Step 10: Validate Dataset

print("Shape :", business_intelligence.shape)

print("Unique Customers :", business_intelligence["customer_unique_id"].nunique())

print("Duplicate Customers :", business_intelligence.duplicated().sum())

Shape : (96096, 25)
Unique Customers : 96096
Duplicate Customers : 0


In [64]:
# investigation
business_intelligence.columns.tolist()

['customer_unique_id',
 'total_orders',
 'delivered_orders',
 'cancelled_orders',
 'first_purchase_date',
 'last_purchase_date',
 'total_spent',
 'average_order_value',
 'maximum_order_value',
 'minimum_order_value',
 'average_payment_installments',
 'preferred_payment_type',
 'recency_days',
 'customer_tenure_days',
 'purchase_frequency',
 'average_purchase_gap',
 'spending_intensity',
 'repeat_customer',
 'cancel_rate',
 'one_time_buyer',
 'high_spender',
 'vip_customer',
 'loyal_customer',
 'at_risk_customer',
 'customer_value_tier']

In [65]:
#Part 9 – Customer 360 Final Merge
#Step 1: Merge All Feature Tables
customer_360 = (
    customer_profile
    .merge(
        purchase_behavior,
        on="customer_unique_id",
        how="left"
    )
    .merge(
        payment_behavior,
        on="customer_unique_id",
        how="left"
    )
    .merge(
        product_behavior,
        on="customer_unique_id",
        how="left"
    )
    .merge(
        experience_behavior,
        on="customer_unique_id",
        how="left"
    )
    .merge(
        business_intelligence.drop(
            columns=[
                "customer_city",
                "customer_state",
                "total_orders",
                "delivered_orders",
                "cancelled_orders",
                "first_purchase_date",
                "last_purchase_date",
                "total_spent",
                "average_order_value",
                "maximum_order_value",
                "minimum_order_value",
                "average_payment_installments",
                "preferred_payment_type"
            ],
            errors="ignore"
        ),
        on="customer_unique_id",
        how="left"
    )
)

In [66]:
#Step 2: Validate Dataset
print("Shape :", customer_360.shape)

print("Unique Customers :", customer_360["customer_unique_id"].nunique())

print("Duplicate Customers :", customer_360.duplicated().sum())

Shape : (96096, 37)
Unique Customers : 96096
Duplicate Customers : 0


In [67]:
#Step 3: Check Missing Values
customer_360.isnull().sum().sort_values(ascending=False)

average_delivery_days     2740
average_delivery_delay    2740
average_review_score       716
unique_products            676
unique_categories          676
                          ... 
cancel_rate                  0
high_spender                 0
vip_customer                 0
loyal_customer               0
at_risk_customer             0
Length: 37, dtype: int64

In [68]:
#Step 4: Preview Dataset
customer_360.head()

,customer_unique_id,customer_city,customer_state,total_orders,delivered_orders,cancelled_orders,first_purchase_date,last_purchase_date,total_spent,average_order_value,maximum_order_value,minimum_order_value,average_payment_installments,preferred_payment_type,total_items_purchased,unique_products,unique_categories,favorite_category,average_items_per_order,average_review_score,average_delivery_days,average_delivery_delay,on_time_delivery_rate,delayed_delivery_rate,recency_days,customer_tenure_days,purchase_frequency,average_purchase_gap,spending_intensity,repeat_customer,cancel_rate,one_time_buyer,high_spender,vip_customer,loyal_customer,at_risk_customer,customer_value_tier
0,861eff4711a542e4b93843c6dd7febb0,franca,SP,1,1,0,2017-05-16 15:05:35,2017-05-16 15:05:35,146.87,146.87,146.87,146.87,2.0,credit_card,1.0,1.0,1.0,moveis_escritorio,1.0,4.0,8.0,-11.0,1.0,0.0,520,0,1.0,0.0,146.87,0,0.0,1,1,0,0,1,Medium
1,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP,1,1,0,2018-01-12 20:48:24,2018-01-12 20:48:24,335.48,335.48,335.48,335.48,8.0,credit_card,1.0,1.0,1.0,utilidades_domesticas,1.0,5.0,16.0,-8.0,1.0,0.0,278,0,1.0,0.0,335.48,0,0.0,1,1,0,0,1,High
2,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP,1,1,0,2018-05-19 16:07:45,2018-05-19 16:07:45,157.73,157.73,157.73,157.73,7.0,credit_card,1.0,1.0,1.0,moveis_escritorio,1.0,5.0,26.0,1.0,0.0,1.0,152,0,1.0,0.0,157.73,0,0.0,1,1,0,0,0,High
3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP,1,1,0,2018-03-13 16:06:38,2018-03-13 16:06:38,173.30,173.30,173.30,173.30,1.0,credit_card,1.0,1.0,1.0,moveis_escritorio,1.0,5.0,14.0,-13.0,1.0,0.0,219,0,1.0,0.0,173.30,0,0.0,1,1,0,0,1,High
4,345ecd01c38d18a9036ed96c73b8d066,campinas,SP,1,1,0,2018-07-29 09:51:30,2018-07-29 09:51:30,252.25,252.25,252.25,252.25,8.0,credit_card,1.0,1.0,1.0,casa_conforto,1.0,5.0,11.0,-6.0,1.0,0.0,81,0,1.0,0.0,252.25,0,0.0,1,1,0,0,0,High


In [ ]:
#Part 10 – Validation & Export
#Step 1: Dataset Information
customer_360.info()

<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 37 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   customer_unique_id            96096 non-null  str           
 1   customer_city                 96096 non-null  str           
 2   customer_state                96096 non-null  str           
 3   total_orders                  96096 non-null  int64         
 4   delivered_orders              96096 non-null  int64         
 5   cancelled_orders              96096 non-null  int64         
 6   first_purchase_date           96096 non-null  datetime64[us]
 7   last_purchase_date            96096 non-null  datetime64[us]
 8   total_spent                   96095 non-null  float64       
 9   average_order_value           96095 non-null  float64       
 10  maximum_order_value           96095 non-null  float64       
 11  minimum_order_value           96095 non

In [ ]:
#investigation
missing_values = (
    customer_360
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_values[missing_values > 0]

average_delivery_days           2740
average_delivery_delay          2740
average_review_score             716
unique_products                  676
unique_categories                676
                                ... 
minimum_order_value                1
average_payment_installments       1
spending_intensity                 1
customer_value_tier                1
total_spent                        1
Length: 16, dtype: int64

In [ ]:
#Step 2: Fill Numeric Missing Values
numeric_columns = customer_360.select_dtypes(include=["int64", "float64"]).columns

customer_360[numeric_columns] = customer_360[numeric_columns].fillna(0)

In [ ]:
#Step 3: Fill String Missing Values
string_columns = [
    "preferred_payment_type",
    "favorite_category"
]

customer_360[string_columns] = customer_360[string_columns].fillna("Unknown")

In [ ]:
#Step 4: Fill customer_value_tier (Category Column)
customer_360["customer_value_tier"] = (
    customer_360["customer_value_tier"]
    .cat.add_categories(["Unknown"])
    .fillna("Unknown")
)

In [ ]:
#Step 5: Verify Missing Values
customer_360.isnull().sum().sum()

np.int64(0)

In [ ]:
#Step 6: Duplicate Check
customer_360.duplicated().sum()

np.int64(0)

In [ ]:
#Step 2: Dataset Validation
print("Shape :", customer_360.shape)

print("Unique Customers :", customer_360["customer_unique_id"].nunique())

print("Duplicate Customers :", customer_360["customer_unique_id"].duplicated().sum())

Shape : (96096, 37)
Unique Customers : 96096
Duplicate Customers : 0


In [ ]:
#Step 3: Missing Value Analysis
missing_values = (
    customer_360
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_values[missing_values > 0]

Series([], dtype: int64)

In [ ]:
#Step 4: Handle Missing Values   Payment Features
payment_columns = [
    "total_spent",
    "average_order_value",
    "maximum_order_value",
    "minimum_order_value",
    "average_payment_installments",
    "spending_intensity"
]

customer_360[payment_columns] = customer_360[payment_columns].fillna(0)

customer_360["preferred_payment_type"] = (
    customer_360["preferred_payment_type"]
    .fillna("Unknown")
)

In [ ]:
#Product Features
product_columns = [
    "total_items_purchased",
    "unique_products",
    "unique_categories",
    "average_items_per_order"
]

customer_360[product_columns] = (
    customer_360[product_columns]
    .fillna(0)
)

customer_360["favorite_category"] = (
    customer_360["favorite_category"]
    .fillna("Unknown")
)

In [ ]:
#Customer Experience Features
customer_360["average_review_score"] = (
    customer_360["average_review_score"]
    .fillna(customer_360["average_review_score"].median())
)

customer_360["average_delivery_days"] = (
    customer_360["average_delivery_days"]
    .fillna(customer_360["average_delivery_days"].median())
)

customer_360["average_delivery_delay"] = (
    customer_360["average_delivery_delay"]
    .fillna(customer_360["average_delivery_delay"].median())
)

In [ ]:
customer_360["customer_value_tier"] = (
    customer_360["customer_value_tier"]
    .fillna("Unknown")
)

In [ ]:
customer_360["customer_value_tier"].cat.categories

Index(['Low', 'Medium', 'High', 'Unknown'], dtype='str')

In [ ]:
#Step 5: Verify Missing Values
print("Total Missing Values :", customer_360.isnull().sum().sum())

print()

print(customer_360.isnull().sum()[customer_360.isnull().sum() > 0])

Total Missing Values : 0

Series([], dtype: int64)


In [ ]:
#Step 6: Duplicate Validation
print("Duplicate Rows :", customer_360.duplicated().sum())

print("Duplicate Customers :",
      customer_360["customer_unique_id"].duplicated().sum())


Duplicate Rows : 0
Duplicate Customers : 0


In [ ]:
#Step 7: Data Type Validation
customer_360.dtypes

customer_unique_id          str
customer_city               str
customer_state              str
total_orders              int64
delivered_orders          int64
                         ...   
high_spender              int64
vip_customer              int64
loyal_customer            int64
at_risk_customer          int64
customer_value_tier    category
Length: 37, dtype: object

In [ ]:
#Step 8: Statistical Summary
customer_360.describe(include="all")

,customer_unique_id,customer_city,customer_state,total_orders,delivered_orders,cancelled_orders,first_purchase_date,last_purchase_date,total_spent,average_order_value,maximum_order_value,minimum_order_value,average_payment_installments,preferred_payment_type,total_items_purchased,unique_products,unique_categories,favorite_category,average_items_per_order,average_review_score,average_delivery_days,average_delivery_delay,on_time_delivery_rate,delayed_delivery_rate,recency_days,customer_tenure_days,purchase_frequency,average_purchase_gap,spending_intensity,repeat_customer,cancel_rate,one_time_buyer,high_spender,vip_customer,loyal_customer,at_risk_customer,customer_value_tier
count,96096,96096,96096,96096.000000,96096.000000,96096.000000,96096,96096,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096,96096.000000,96096.000000,96096.000000,96096,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096.000000,96096
unique,96096,4118,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,74,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
top,861eff4711a542e4b93843c6dd7febb0,sao paulo,SP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,credit_card,NaN,NaN,NaN,cama_mesa_banho,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low
freq,1,14971,40295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73531,NaN,NaN,NaN,8998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32034
mean,NaN,NaN,NaN,1.034809,1.003975,0.006504,2017-12-30 19:19:10.429206,2018-01-02 12:40:19.655865,166.592492,158.707135,161.391976,156.143847,2.901417,NaN,1.172265,1.061303,1.018929,NaN,1.131087,4.054553,11.758486,-11.511088,0.904377,0.095623,288.735691,2.711507,0.989322,1.275936,161.400117,0.031188,0.005985,0.968812,0.499844,0.027410,0.003517,0.711289,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25%,NaN,NaN,NaN,1.000000,1.000000,0.000000,2017-09-11 19:52:06,2017-09-15 09:04:17.250000,63.120000,60.850000,61.820000,58.377500,1.000000,NaN,1.000000,1.000000,1.000000,NaN,1.000000,4.000000,6.000000,-17.000000,1.000000,0.000000,164.000000,0.000000,1.000000,0.000000,62.457500,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,NaN
50%,NaN,NaN,NaN,1.000000,1.000000,0.000000,2018-01-18 13:33:08,2018-01-21 19:39:16,108.000000,103.750000,105.380000,101.740000,2.000000,NaN,1.000000,1.000000,1.000000,NaN,1.000000,5.000000,10.000000,-12.000000,1.000000,0.000000,269.000000,0.000000,1.000000,0.000000,105.825000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,NaN
75%,NaN,NaN,NaN,1.000000,1.000000,0.000000,2018-05-04 10:38:45,2018-05-06 20:14:49.750000,183.530000,175.080000,177.560000,173.270000,4.000000,NaN,1.000000,1.000000,1.000000,NaN,1.000000,5.000000,15.000000,-7.000000,1.000000,0.000000,398.000000,0.000000,1.000000,0.000000,177.210000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,NaN
max,NaN,NaN,NaN,17.000000,15.000000,3.000000,2018-10-17 17:30:18,2018-10-17 17:30:18,13664.080000,13664.080000,13664.080000,13664.080000,24.000000,NaN,24.000000,15.000000,5.000000,NaN,21.000000,5.000000,209.000000,188.000000,1.000000,1.000000,773.000000,633.000000,6.000000,304.000000,13664.080000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,NaN


In [ ]:
#Step 9: Preview Final Dataset
customer_360.head()

,customer_unique_id,customer_city,customer_state,total_orders,delivered_orders,cancelled_orders,first_purchase_date,last_purchase_date,total_spent,average_order_value,maximum_order_value,minimum_order_value,average_payment_installments,preferred_payment_type,total_items_purchased,unique_products,unique_categories,favorite_category,average_items_per_order,average_review_score,average_delivery_days,average_delivery_delay,on_time_delivery_rate,delayed_delivery_rate,recency_days,customer_tenure_days,purchase_frequency,average_purchase_gap,spending_intensity,repeat_customer,cancel_rate,one_time_buyer,high_spender,vip_customer,loyal_customer,at_risk_customer,customer_value_tier
0,861eff4711a542e4b93843c6dd7febb0,franca,SP,1,1,0,2017-05-16 15:05:35,2017-05-16 15:05:35,146.87,146.87,146.87,146.87,2.0,credit_card,1.0,1.0,1.0,moveis_escritorio,1.0,4.0,8.0,-11.0,1.0,0.0,520,0,1.0,0.0,146.87,0,0.0,1,1,0,0,1,Medium
1,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP,1,1,0,2018-01-12 20:48:24,2018-01-12 20:48:24,335.48,335.48,335.48,335.48,8.0,credit_card,1.0,1.0,1.0,utilidades_domesticas,1.0,5.0,16.0,-8.0,1.0,0.0,278,0,1.0,0.0,335.48,0,0.0,1,1,0,0,1,High
2,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP,1,1,0,2018-05-19 16:07:45,2018-05-19 16:07:45,157.73,157.73,157.73,157.73,7.0,credit_card,1.0,1.0,1.0,moveis_escritorio,1.0,5.0,26.0,1.0,0.0,1.0,152,0,1.0,0.0,157.73,0,0.0,1,1,0,0,0,High
3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP,1,1,0,2018-03-13 16:06:38,2018-03-13 16:06:38,173.30,173.30,173.30,173.30,1.0,credit_card,1.0,1.0,1.0,moveis_escritorio,1.0,5.0,14.0,-13.0,1.0,0.0,219,0,1.0,0.0,173.30,0,0.0,1,1,0,0,1,High
4,345ecd01c38d18a9036ed96c73b8d066,campinas,SP,1,1,0,2018-07-29 09:51:30,2018-07-29 09:51:30,252.25,252.25,252.25,252.25,8.0,credit_card,1.0,1.0,1.0,casa_conforto,1.0,5.0,11.0,-6.0,1.0,0.0,81,0,1.0,0.0,252.25,0,0.0,1,1,0,0,0,High


In [ ]:
# ============================================================
# FINAL CUSTOMER 360 VALIDATION
# ============================================================

print("=" * 60)
print("CUSTOMER 360 DATASET VALIDATION")
print("=" * 60)

# Shape
print(f"\nDataset Shape : {customer_360.shape}")

# Unique Customers
print(f"Unique Customers : {customer_360['customer_unique_id'].nunique()}")

# Duplicate Customers
print(f"Duplicate Customers : {customer_360['customer_unique_id'].duplicated().sum()}")

# Duplicate Rows
print(f"Duplicate Rows : {customer_360.duplicated().sum()}")

# Duplicate Columns
duplicate_columns = customer_360.columns[customer_360.columns.duplicated()]

print(f"Duplicate Columns : {len(duplicate_columns)}")

if len(duplicate_columns) > 0:
    print(duplicate_columns)

# Missing Values
missing = customer_360.isnull().sum()

print("\nMissing Values")
print("-" * 60)

if missing.sum() == 0:
    print("No Missing Values Found")
else:
    print(missing[missing > 0])

# Data Types
print("\nData Types")
print("-" * 60)
print(customer_360.dtypes.value_counts())

print("\nDataset Preview")
print("-" * 60)
display(customer_360.head())

print("\nValidation Completed Successfully!")

CUSTOMER 360 DATASET VALIDATION

Dataset Shape : (96096, 37)
Unique Customers : 96096
Duplicate Customers : 0
Duplicate Rows : 0
Duplicate Columns : 0

Missing Values
------------------------------------------------------------
No Missing Values Found

Data Types
------------------------------------------------------------
float64           18
int64             11
str                5
datetime64[us]     2
category           1
Name: count, dtype: int64

Dataset Preview
------------------------------------------------------------


,customer_unique_id,customer_city,customer_state,total_orders,delivered_orders,cancelled_orders,first_purchase_date,last_purchase_date,total_spent,average_order_value,maximum_order_value,minimum_order_value,average_payment_installments,preferred_payment_type,total_items_purchased,unique_products,unique_categories,favorite_category,average_items_per_order,average_review_score,average_delivery_days,average_delivery_delay,on_time_delivery_rate,delayed_delivery_rate,recency_days,customer_tenure_days,purchase_frequency,average_purchase_gap,spending_intensity,repeat_customer,cancel_rate,one_time_buyer,high_spender,vip_customer,loyal_customer,at_risk_customer,customer_value_tier
0,861eff4711a542e4b93843c6dd7febb0,franca,SP,1,1,0,2017-05-16 15:05:35,2017-05-16 15:05:35,146.87,146.87,146.87,146.87,2.0,credit_card,1.0,1.0,1.0,moveis_escritorio,1.0,4.0,8.0,-11.0,1.0,0.0,520,0,1.0,0.0,146.87,0,0.0,1,1,0,0,1,Medium
1,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP,1,1,0,2018-01-12 20:48:24,2018-01-12 20:48:24,335.48,335.48,335.48,335.48,8.0,credit_card,1.0,1.0,1.0,utilidades_domesticas,1.0,5.0,16.0,-8.0,1.0,0.0,278,0,1.0,0.0,335.48,0,0.0,1,1,0,0,1,High
2,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP,1,1,0,2018-05-19 16:07:45,2018-05-19 16:07:45,157.73,157.73,157.73,157.73,7.0,credit_card,1.0,1.0,1.0,moveis_escritorio,1.0,5.0,26.0,1.0,0.0,1.0,152,0,1.0,0.0,157.73,0,0.0,1,1,0,0,0,High
3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP,1,1,0,2018-03-13 16:06:38,2018-03-13 16:06:38,173.30,173.30,173.30,173.30,1.0,credit_card,1.0,1.0,1.0,moveis_escritorio,1.0,5.0,14.0,-13.0,1.0,0.0,219,0,1.0,0.0,173.30,0,0.0,1,1,0,0,1,High
4,345ecd01c38d18a9036ed96c73b8d066,campinas,SP,1,1,0,2018-07-29 09:51:30,2018-07-29 09:51:30,252.25,252.25,252.25,252.25,8.0,credit_card,1.0,1.0,1.0,casa_conforto,1.0,5.0,11.0,-6.0,1.0,0.0,81,0,1.0,0.0,252.25,0,0.0,1,1,0,0,0,High



Validation Completed Successfully!


In [ ]:
#Step 10: Export Final Dataset
customer_360.to_csv(
    "../1 data/02_processed data/customer_360.csv",
    index=False
)

print("Customer 360 dataset exported successfully.")

Customer 360 dataset exported successfully.
